# Retrieval of the data


# Important Concepts


### Understanding Cosine distance and Similarity

- [2 Min Video](https://youtu.be/zcUGLp5vwaQ)

![Credits: ML-science.com](https://images.squarespace-cdn.com/content/v1/5acbdd3a25bf024c12f4c8b4/1601843298424-0XDCB1FGXHDPIZM4TBML/Cosine+Similarity.png?format=2500w)

![](./images/vectors.png)

![Credits: ML-science.com](https://images.squarespace-cdn.com/content/v1/5acbdd3a25bf024c12f4c8b4/1601861231635-6FB41TNOOPTVI7RXT7DF/Cosine+Similarity+Radial+Examples.png?format=2500w)


## Setup OpenAI API


In [7]:
import os

import azure.identity
import dotenv
import openai

# Set up OpenAI client based on environment variables
dotenv.load_dotenv()
print("Loaded environment variables from .env file")
print(dotenv.dotenv_values(".env"))
AZURE_OPENAI_SERVICE = os.getenv("AZURE_OPENAI_SERVICE")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_TEXT_COMPLETION_MODEL = os.getenv("AZURE_OPENAI_TEXT_COMPLETION_MODEL")

azure_credential = azure.identity.AzureDeveloperCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID"))
token_provider = azure.identity.get_bearer_token_provider(
    azure_credential, "https://cognitiveservices.azure.com/.default"
)
openai_client = openai.AzureOpenAI(
    api_version="2024-06-01",
    azure_endpoint=f"https://{AZURE_OPENAI_SERVICE}.openai.azure.com",
    azure_ad_token_provider=token_provider,
)

Loaded environment variables from .env file
OrderedDict({'AZURE_OPENAI_SERVICE': 'ai-prateek4732ai561893487136', 'AZURE_OPENAI_TEXT_COMPLETION_MODEL': 'gpt-4.1-nano', 'AZURE_OPENAI_EMBEDDING_MODEL': 'text-embedding-3-small', 'AZURE_TENANT_ID': '06f18712-6c3a-4b61-9475-bf2c226971b3'})


### Similar meanings have almost same cosine distance


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

sentences1 = [
    "The new movie is awesome",
    "The new movie is awesome",
    "The new movie is awesome",
    "The new movie is awesome",
]

sentences2 = [
    "The new movie is awesome",
    "This recent movie is so good",
    "sskndsjkcnjdscnsjdcsdcsdcsdc",
    "9832u498u329",
]


def get_embeddings(sentences):
    embeddings_response = openai_client.embeddings.create(model=AZURE_OPENAI_EMBEDDING_MODEL, input=sentences)
    return [np.array(obj.embedding) for obj in embeddings_response.data]


embeddings1 = get_embeddings(sentences1)
embeddings2 = get_embeddings(sentences2)

# Collect results
rows = []
for i in range(len(sentences1)):
    emb1 = embeddings1[i].reshape(1, -1)
    emb2 = embeddings2[i].reshape(1, -1)
    score = cosine_similarity(emb1, emb2)[0][0]
    rows.append([sentences1[i], sentences2[i], round(score, 4)])

# Create and print table
df = pd.DataFrame(rows, columns=["Sentence 1", "Sentence 2", "Cosine Similarity"])
print(df.to_string(index=False))

              Sentence 1                   Sentence 2  Cosine Similarity
The new movie is awesome     The new movie is awesome             1.0000
The new movie is awesome This recent movie is so good             0.5638
The new movie is awesome sskndsjkcnjdscnsjdcsdcsdcsdc             0.1255
The new movie is awesome                 9832u498u329             0.0667


# Data Retrieval


### Similarity Search on Movies Index


In [10]:
import faiss
import numpy as np
import json
import pandas as pd
from utils import get_title, get_meta

# -----------------------
# Load FAISS + metadata
# -----------------------
index = faiss.read_index("movie_title_embeddings_cosine.index")

with open("movies.json", "r") as f:
    movies = json.load(f)["movies"]

assert index.ntotal == len(movies), f"Vector count {index.ntotal} != movies {len(movies)}"

# -----------------------
# Define query
# -----------------------
# query = "A story about prisoner"
query = {
    "title": "Forrest Gump",
    "year": "1994",
    "genres": ["Comedy", "Drama"],
    "plot": "Forrest Gump, while not intelligent, has accidentally been present at many historic moments, but his true love, Jenny Curran, eludes him.",
}

resp = openai_client.embeddings.create(model=AZURE_OPENAI_EMBEDDING_MODEL, input=[str(query)])
q = np.array(resp.data[0].embedding, dtype="float32").reshape(1, -1)

# Normalize query (because index is normalized for cosine)
faiss.normalize_L2(q)

# -----------------------
# Search top-k
# -----------------------
k = 5
scores, ids = index.search(q, k)  # cosine similarity in [0,1]

rows = []
for r, (doc_id, cos) in enumerate(zip(ids[0], scores[0]), start=1):
    m = movies[int(doc_id)]
    year, genres = get_meta(m)
    rows.append(
        {
            "Rank": r,
            "Doc ID": int(doc_id),
            "Title": get_title(m),
            "Year": year,
            "Genres": genres,
            "Cosine": round(float(cos), 3),
        }
    )

df = pd.DataFrame(rows, columns=["Rank", "Doc ID", "Title", "Year", "Genres", "Cosine"])
print(df.to_string(index=False))

 Rank  Doc ID                    Title Year                   Genres  Cosine
    1       2             Forrest Gump 1994            Comedy, Drama   0.855
    2       1             Pulp Fiction 1994             Crime, Drama   0.429
    3       0 The Shawshank Redemption 1994             Crime, Drama   0.336
    4      13               Fight Club 1999                    Drama   0.336
    5      10                Gladiator 2000 Action, Adventure, Drama   0.333


### Visualize Vectors and Distance in 3D


In [11]:
import json
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import openai  # Make sure openai client is configured
import time

# Load embeddings
json_file = "/Users/prateek/workspace/repo/rag-with-azure-ai-search-notebooks/openai_movies.json"

with open(json_file, "r") as f:
    movie_vectors = json.load(f)

print(f"Loaded {len(movie_vectors)} movie embeddings.")

# Set query keyword
query = "king"  # You can replace this with input() for interactive use

# Filter movies by keyword in title
filtered_titles = [title for title in movie_vectors if query.lower() in title.lower()]
filtered_vectors = [movie_vectors[title] for title in filtered_titles]

print(f"Found {len(filtered_titles)} movies containing '{query}'.")

if len(filtered_titles) == 0:
    print("No movies found with this keyword.")
else:
    # Generate query embedding
    print("Generating query embedding...")
    query_embedding_response = openai_client.embeddings.create(model="text-embedding-ada-002", input=[query])
    query_vector = query_embedding_response.data[0].embedding

    # Compute cosine similarities
    print("Computing similarities...")
    similarities = cosine_similarity([query_vector], filtered_vectors)[0]

    # Build DataFrame with results
    df = pd.DataFrame({"Movie": filtered_titles, "Score": similarities}).sort_values("Score", ascending=False)

    print("Top similar movies:")
    display(df.head(10))

    # Prepare 3D visualization
    print("Creating 3D visualization...")

    if len(filtered_vectors) >= 3:
        pca = PCA(n_components=3)
        reduced = pca.fit_transform(filtered_vectors)
    elif len(filtered_vectors) == 2:
        reduced = np.hstack([np.array(filtered_vectors), np.zeros((2, 1))])
    elif len(filtered_vectors) == 1:
        reduced = np.hstack([np.array(filtered_vectors), np.zeros((1, 2))])
    else:
        reduced = np.zeros((0, 3))

    time.sleep(1)  # Delay to simulate loading

    if len(filtered_vectors):
        df_plot = pd.DataFrame(
            {
                "x": reduced[:, 0] if len(reduced) else [],
                "y": reduced[:, 1] if len(reduced) else [],
                "z": reduced[:, 2] if len(reduced) else [],
                "Movie": filtered_titles,
                "Score": similarities,
            }
        )

        fig = px.scatter_3d(
            df_plot,
            x="x",
            y="y",
            z="z",
            text="Movie",
            hover_data=["Movie", "Score"],
            color="Score",
            color_continuous_scale="Viridis",
            title=f"3D Vector visualization for '{query}'",
        )
        fig.update_traces(marker=dict(size=5))
        fig.show()
    else:
        print("Not enough data to create visualization.")

Loaded 573 movie embeddings.
Found 7 movies containing 'king'.
Generating query embedding...
Computing similarities...
Top similar movies:


,Movie,Score
5,Monkey Kingdom,0.841490
3,King Arthur,0.830703
2,A Kid in King Arthur's Court,0.825370
1,The Lion King,0.821380
4,Waking Sleeping Beauty,0.793063
6,Alice Through the Looking Glass,0.769401
0,Taking Care of Business,0.768853


Creating 3D visualization...
